# 02 · Decoding one token at a time
Level L1 · Part 1 · core · ~20 min · builds on 01

## Goal
By the end of this lesson you can explain the autoregressive loop, count the attention work for a prompt and for each new token, and say why a long context adds attention work to the first token and to every later token.

## Why it matters
- **Without it:** you treat each token as the same unit of work. A 300-token chat and a 30,000-token document would look like the same job per token, and your latency plan would miss the long one.
- **With it:** you can say where the attention work goes. Prefill attention work grows with the square of the prompt length, so for long prompts the first token is slower. Each decode step grows with the length of the context so far.
- **When you don't need it:** if every request is short and you measure latency per request, the counting is optional. Lesson 07 measures it instead.

## The idea
A language model writes an answer the way you might read a long sentence aloud one word at a time, looking back at everything said so far before each word. This loop is **autoregressive decoding**: each new **token** depends on all the tokens before it. The core operation is **attention**: each token compares itself with the earlier tokens to decide what to take from them. Each token produces a **query** (what it is looking for) and a **key** and a **value** (what it offers to later tokens). A token compares its query with the earlier tokens' keys, then takes a mix of their values. Every comparison is one **query-key pair**. Counting the pairs counts the attention work, and that count grows fast as the context grows.

## Picture
The prompt is processed once in parallel. Its keys and values go into the **KV cache**, the stored keys and values of earlier tokens. Then each decode step compares one new token with everything in the cache, and adds that token's keys and values to it.

```mermaid
flowchart LR
  P["Prompt: n tokens"] --> F["Prefill: about n x n/2 pairs, one parallel pass"]
  F --> C["KV cache: keys and values of the tokens so far"]
  C --> D["Decode step: 1 new query compares with all cached keys"]
  D -->|"append this token's keys and values"| C
```

## Explained
**Prefill** is the one parallel pass over the whole prompt. **Decode** is the step-by-step generation after it, one token per step. The **KV cache** is the stored keys and values, so they are not recomputed.

**Prefill: every prompt token looks back.** Attention is causal: a token may look at itself and at earlier tokens, never at later ones. So token 1 makes 1 pair, token 2 makes 2 pairs, and so on. For a prompt of n tokens the total is n(n+1)/2, about n squared over 2. Double the prompt and the count roughly quadruples.

**Decode: one new token looks back at everything.** After the prompt, each step has one query (the new token) that compares with all cached tokens. With a context of n tokens, that is about n pairs. Each step costs more than the last, because the cache grows by one token per step. But the growth is linear, not squared.


## Explained (continued)
**Why the cache helps.** Without it, step i would recompute the keys and values of all the earlier tokens. The cache stores them, so only the new token is computed. The attention itself still compares the new token with every cached token. The cache removes the recomputation. It does not remove the comparisons.

**Other work.** The weight matrix multiplications handle each token once, so their work grows linearly with the number of tokens. At short contexts they are most of the work. Attention takes a larger share as the context grows. Lesson 06 puts numbers on that balance.

## Worked example: the same pair counts for different prompt sizes
The next cell counts the pairs. It does not time anything. Prefill uses the causal count. A decode step with context n uses n pairs. The answer is 200 tokens long.

In [1]:
def prefill_pairs(n):
    """Query-key pairs for a prompt of n tokens: token i looks at tokens 1..i."""
    return n * (n + 1) // 2

def decode_pairs(prompt, answer):
    """With a KV cache: step i compares its one query with prompt + i keys (including itself)."""
    return sum(prompt + i for i in range(1, answer + 1))

ANSWER = 200
print(f"{'prompt':>8} {'prefill pairs':>15} {'decode pairs':>14}")
for prompt in (1_000, 2_000, 4_000, 8_000):
    print(f"{prompt:>8,} {prefill_pairs(prompt):>15,} {decode_pairs(prompt, ANSWER):>14,}")

  prompt   prefill pairs   decode pairs
   1,000         500,500        220,100
   2,000       2,001,000        420,100
   4,000       8,002,000        820,100
   8,000      32,004,000      1,620,100


**What you should see:** doubling the prompt multiplies the prefill pairs by about 4 (500,500 to 2,001,000), but multiplies the decode pairs for 200 tokens by less than 2 (220,100 to 420,100). Prefill grows with the square. Decode grows with the context, one step at a time.

Now the Brightside case from lesson 01: a 1,600-token prompt (a 1,500-token policy plus a 100-token question) and a 200-token answer. The last number needs one more idea. Without a cache, each step reruns the whole sequence, including the new token, so each step pays a full causal pass again.

In [2]:
PROMPT = 1_600   # the Brightside prompt from lesson 01

def recompute_pairs(prompt, answer):
    """Without a cache: step i reruns the whole sequence, including the new token (prompt + i tokens)."""
    return sum(prefill_pairs(prompt + i) for i in range(1, answer + 1))

prefill = prefill_pairs(PROMPT)
cached = decode_pairs(PROMPT, ANSWER)
recomputed = recompute_pairs(PROMPT, ANSWER)
print(f"prefill, one pass over the prompt:   {prefill:,} pairs")
print(f"decode with KV cache, 200 steps:     {cached:,} pairs")
print(f"decode without cache, 200 steps:     {recomputed:,} pairs")
print(f"ratio without / with cache:          {recomputed / cached:,.0f}x (counted, not measured)")

prefill, one pass over the prompt:   1,280,800 pairs
decode with KV cache, 200 steps:     340,100 pairs
decode without cache, 200 steps:     289,673,400 pairs
ratio without / with cache:          852x (counted, not measured)


**What you should see:** the prefill is the largest single block for this prompt, about 1.28 million pairs. The 200 decode steps with the cache add about 340,000 pairs. Without the cache, the same answer needs about 290 million pairs, roughly 852 times as many. The cache is what keeps the answer cheap. The counts are estimates of attention work only, not timings.

## At work
- **Character.AI** reports serving cost down at least 33 times since 2022, and lists inter-turn KV caching (reusing earlier turns' keys and values) among its techniques. → lesson 14 (prefix caching across a shared prompt).
- **Cloudflare (May 2026)** reports more than 56 GiB of KV-cache room on 2 H200 GPUs for Llama 4 Scout, which supports over 1.2 million tokens of context. Long context is paid in cache memory on every request that uses it. → lessons 03 (KV cache size from a config) and 05 (concurrency from the memory budget).
- **Google Cloud (GKE)** advises scaling inference on queue size or batch size, not GPU utilisation. A 200-token request and a 20,000-token request are not the same load, so counting requests is not enough. → lesson 31 (scaling on queue depth and batch size).

**Sources:** Character.AI, "Optimizing AI inference at Character.AI": https://blog.character.ai/optimizing-ai-inference-at-character-ai/ · Cloudflare, "High-performance LLMs": https://blog.cloudflare.com/high-performance-llms/ · Google Cloud, "Tuning the GKE HPA to run inference on GPUs": https://cloud.google.com/blog/products/containers-kubernetes/tuning-the-gke-hpa-to-run-inference-on-gpus

## Common misconceptions
1. **"Each token costs the same."** Not in attention. A decode step's work grows with the cached context, and prefill grows with the square of the prompt.
2. **"The KV cache makes attention free."** It removes the recomputation of old keys and values. Each new token still compares with every cached token, and the cache must stay in GPU memory until the request ends.
3. **"Long context is only a memory problem."** It is a time problem too. Prefill attention work grows with the square of the prompt, so for long prompts the first token is slower. Each decode step also does more attention work as the cache grows, so later tokens cost more. Memory is covered in lessons 03 and 05.

## Check yourself
1. Brightside's policy prompt grows from 1,500 to 3,000 tokens. By about how much does the prefill pair count grow? Doubling the prompt gives about 4x; check the numbers with `prefill_pairs` from the worked example if you like.
2. A chat has 4,000 tokens of history. Compare the attention work of decode step 1 with step 1,000. Which is bigger, and by about how much?
3. Why does the KV cache not remove attention cost, even though it removes recomputation?
4. Your p99 time to first token doubles when a few users send 20,000-token documents. Which phase explains that, and what else can make it worse?
5. Your provider bills per token, so you assume a 10,000-token request costs 10 times a 1,000-token one. What does the attention count say about your own server?

**Answers** — try to answer first, then expand the hidden cell below.

1. About 4 times. 1,500 tokens give 1,125,750 pairs; 3,000 tokens give 4,501,500 pairs.
2. Step 1,000 is bigger. Step 1 compares with 4,001 keys, and step 1,000 with 5,000 keys, so about 1.25 times the work.
3. The cache stores old keys and values so they are not recomputed. Each new token still has to compare its query with every stored key, so the comparisons still happen, and the cache takes memory.
4. Prefill. A 20,000-token prompt makes about 200 million pairs, which is quadratic. Long prefills also make short requests wait behind them (lesson 12 shows how to split them up).
5. The attention pairs for prefill rise about 100 times (50,005,000 against 500,500), not 10 times. Price per token hides that. Measure your own server (lessons 06 and 07).

## Recap
- A request has a **prefill** (the whole prompt in one parallel pass) and then **decode** (one token per step).
- **Prefill attention** compares every prompt token with the earlier ones, so its pairs grow with the square of the prompt: n(n+1)/2.
- **Each decode step** compares one new token with every cached token, so its work grows linearly with the context.
- The **KV cache** saves recomputing old keys and values. It does not remove the comparisons, and it holds GPU memory until the request ends.

**Interview:** "Why does a long prompt slow the first token more than the later tokens?" Strong answer: prefill compares every prompt token with every earlier token, so its work grows with the square of the prompt length; each decode step compares one token with the cache, so its work grows linearly; both need the KV cache in memory.